In [1]:
import os
import sys
import torch

print("=== ENVIRONMENT CHECK ===")
print("Current folder before fix:", os.getcwd())
print("Python:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

# Force notebook work into the existing dl_mnist folder
workdir = os.path.join(os.path.expanduser("~"), "dl_mnist")
os.makedirs(workdir, exist_ok=True)
os.chdir(workdir)

print("\n=== WORKING DIRECTORY FIXED ===")
print("Now working in:", os.getcwd())
print("Files here:", os.listdir("."))

=== ENVIRONMENT CHECK ===
Current folder before fix: /users/khalessi
Python: /usr/bin/python3.12
Torch version: 2.9.1+cu129
CUDA available: True
GPU name: Tesla V100-SXM2-32GB

=== WORKING DIRECTORY FIXED ===
Now working in: /users/khalessi/dl_mnist
Files here: ['results_fresh.csv', 'download_mnist.py', 'results.csv', 'train_mlp_depth.py', 'data']


# Assignment 1 — MNIST MLP Depth Comparison

This notebook uses PyTorch on Puhti to compare MLP performance on the MNIST dataset using 3, 5, and 10 hidden layers.

In [ ]:
from torchvision import datasets, transforms
import os

transform = transforms.Compose([
    transforms.ToTensor()
])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

print("Training samples:", len(train_ds))
print("Test samples:", len(test_ds))
print("Data saved in:", os.path.abspath("./data"))

Training samples: 60000
Test samples: 10000
Data saved in: /users/khalessi/dl_mnist/data


## Model setup

We use a fully connected neural network (MLP) with:
- hidden width = 256
- optimizer = Adam
- learning rate = 0.001
- batch size = 128
- epochs = 10

Only the number of hidden layers changes: 3, 5, and 10.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds = datasets.MNIST(root="./data", train=True, download=False, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=False, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

class MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, hidden_layers=3, num_classes=10):
        super().__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())

        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())

        layers.append(nn.Linear(hidden_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return correct / total

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def run_experiment(hidden_layers, epochs=10, hidden_dim=256, lr=1e-3):
    model = MLP(hidden_dim=hidden_dim, hidden_layers=hidden_layers).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    best_acc = 0.0
    epoch_times = []

    print(f"\n===== Running model with {hidden_layers} hidden layers =====")
    print("Parameter count:", count_params(model))

    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

        sec = time.time() - t0
        epoch_times.append(sec)

        acc = evaluate(model, test_loader)
        best_acc = max(best_acc, acc)

        print(f"Epoch {epoch:02d}/{epochs} | {sec:.2f}s | test_acc = {acc*100:.2f}%")

    final_acc = evaluate(model, test_loader)

    return {
        "layers": hidden_layers,
        "hidden": hidden_dim,
        "params": count_params(model),
        "best_test_acc": best_acc,
        "final_test_acc": final_acc,
        "sec_per_epoch": sum(epoch_times) / len(epoch_times),
        "total_sec": sum(epoch_times),
        "epochs": epochs,
        "batch": 128,
        "lr": lr,
        "device": str(device)
    }

Using device: cuda


In [ ]:
results = []

for layers in [3, 5, 10]:
    result = run_experiment(hidden_layers=layers, epochs=10)
    results.append(result)

results


===== Running model with 3 hidden layers =====
Parameter count: 335114
Epoch 01/10 | 4.66s | test_acc = 96.23%
Epoch 02/10 | 4.56s | test_acc = 97.39%
Epoch 03/10 | 4.56s | test_acc = 97.64%
Epoch 04/10 | 4.60s | test_acc = 97.37%
Epoch 05/10 | 4.56s | test_acc = 97.61%
Epoch 06/10 | 4.59s | test_acc = 97.31%
Epoch 07/10 | 4.58s | test_acc = 97.52%
Epoch 08/10 | 4.62s | test_acc = 97.84%
Epoch 09/10 | 4.57s | test_acc = 97.93%
Epoch 10/10 | 4.54s | test_acc = 97.31%

===== Running model with 5 hidden layers =====
Parameter count: 466698
Epoch 01/10 | 4.48s | test_acc = 95.88%
Epoch 02/10 | 4.58s | test_acc = 97.20%
Epoch 03/10 | 4.61s | test_acc = 97.30%
Epoch 04/10 | 4.57s | test_acc = 97.46%
Epoch 05/10 | 4.56s | test_acc = 97.51%
Epoch 06/10 | 4.56s | test_acc = 97.72%
Epoch 07/10 | 4.53s | test_acc = 97.30%
Epoch 08/10 | 4.58s | test_acc = 98.03%
Epoch 09/10 | 4.59s | test_acc = 97.84%
Epoch 10/10 | 4.62s | test_acc = 97.85%

===== Running model with 10 hidden layers =====
Paramet

[{'layers': 3,
  'hidden': 256,
  'params': 335114,
  'best_test_acc': 0.9793,
  'final_test_acc': 0.9731,
  'sec_per_epoch': 4.583692479133606,
  'total_sec': 45.83692479133606,
  'epochs': 10,
  'batch': 128,
  'lr': 0.001,
  'device': 'cuda'},
 {'layers': 5,
  'hidden': 256,
  'params': 466698,
  'best_test_acc': 0.9803,
  'final_test_acc': 0.9785,
  'sec_per_epoch': 4.568125367164612,
  'total_sec': 45.68125367164612,
  'epochs': 10,
  'batch': 128,
  'lr': 0.001,
  'device': 'cuda'},
 {'layers': 10,
  'hidden': 256,
  'params': 795658,
  'best_test_acc': 0.9775,
  'final_test_acc': 0.9773,
  'sec_per_epoch': 4.579050278663635,
  'total_sec': 45.79050278663635,
  'epochs': 10,
  'batch': 128,
  'lr': 0.001,
  'device': 'cuda'}]

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df

,layers,hidden,params,best_test_acc,final_test_acc,sec_per_epoch,total_sec,epochs,batch,lr,device
0,3,256,335114,0.9793,0.9731,4.583692,45.836925,10,128,0.001,cuda
1,5,256,466698,0.9803,0.9785,4.568125,45.681254,10,128,0.001,cuda
2,10,256,795658,0.9775,0.9773,4.579050,45.790503,10,128,0.001,cuda


In [ ]:
import os

csv_path = os.path.join(os.getcwd(), "results.csv")
df.to_csv(csv_path, index=False)

print("Saved CSV to:", csv_path)

Saved CSV to: /users/khalessi/dl_mnist/results.csv


## Discussion

The results show that the 5-layer MLP performed best in this experiment, reaching the highest best and final test accuracy.

The training times for the 3-layer, 5-layer, and 10-layer models were very similar in this GPU-based run, so increasing depth did not create a major runtime penalty in this case.

At the same time, adding more depth did not always improve performance. The 10-layer model did not outperform the 5-layer model, which shows that a deeper network is not automatically better. In this experiment, the 5-layer model gave the best overall result.

## Conclusion

In this assignment, PyTorch was used on Puhti with GPU support to train MLP models on MNIST using 3, 5, and 10 hidden layers.

The results showed that the 5-layer model achieved the best performance in this experiment. This notebook demonstrates how model depth can affect accuracy and training behavior, and it shows that the best choice is not always the deepest model.